# Learning Objectives

- Build an LLM assistant for document-based Q&A using retrieval-augmented generation.

In [1]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime


c:\Users\Rohit kumar\Desktop\assign - Copy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_folder_location = "tesla-annual-reports"

In [3]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [4]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [5]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [6]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [7]:
len(tesla_10k_chunks)

3337

In [8]:
print(tesla_10k_chunks[0].page_content)

UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA
The	Nasdaq	Gl

In [9]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Rohit kumar\AppData\Local\Temp\ipykernel_13548\726496232.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [12]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [13]:
chromadb_client.count_collections()

1

In [14]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [15]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023']

In [16]:
print(tesla_10k_chunks[0].page_content)

UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA
The	Nasdaq	Gl

In [ ]:
i = 0 # Initialize the starting index for the chunks

while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
    vectorstore.add_documents( # Add documents to the vector store in batches of 500
        documents=tesla_10k_chunks[i:i+500], # Get the current batch of 500 chunks
        ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
    )

    i += 500 # Increment the index by 500 to move to the next batch
    time.sleep(5) # Pause for 5 seconds to avoid rate limiting issues with the vector store

In [17]:
import chromadb

from langchain_chroma import Chroma



In [18]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [19]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001F49109CD90>, search_kwargs={'k': 5})

In [20]:
user_query = "What was the automotive revenue in 2021?"

##Hypothetical Questions

In [21]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
from groq import Groq
client = Groq()

In [ ]:
#from openai import OpenAI



In [24]:
#
#client = OpenAI(
 ##  base_url="https://integrate.api.nvidia.com/v1"


In [25]:
model_name = 'openai/gpt-oss-120b'

In [33]:

user_message_template = """
<Document>
{documents}
</Document>
"""

In [34]:
tesla_10k_chunks

[Document(metadata={'producer': 'Qt 5.11.3', 'creator': 'wkhtmltopdf 0.12.5', 'creationdate': '2022-05-02T10:10:26+00:00', 'title': '', 'source': 'tesla-annual-reports\\tsla-10ka_20211231-gen.pdf', 'total_pages': 56, 'page': 0, 'page_label': '1'}, page_content='UNITED\tSTATES\nSECURITIES\tAND\tEXCHANGE\tCOMMISSION\nWashington,\tD.C.\t20549\n\t\nFORM\t\n10-K/A\n(Amendment\tNo.\t1)\n\t\n(Mark\tOne)\n☒\nANNUAL\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t1934\nFor\tthe\tfiscal\tyear\tended\t\nDecember\t31,\t\n2021\n\t\nOR\n☐\nTRANSITION\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t1934\nFor\tthe\ttransition\tperiod\tfrom\t\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\n\tto\t\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\nCommission\tFile\tNumber:\t\n001-34756\n\t\nTesla,\tInc.\n(Exact\tname\tof\tregistrant\tas\tspecified\tin\tits\tcharter)\n\t\n\t\nDelaware\n\t\n91-2197729\n(State\tor\tother\tjurisdiction\tof\n

In [36]:
BATCH_SIZE = 100

chunk_batches = []

for i in range(0, len(tesla_10k_chunks), BATCH_SIZE):

    batch = tesla_10k_chunks[i:i+BATCH_SIZE]

    # Token limit control
    batch_text = "\n\n".join(
        [doc.page_content[:200] for doc in batch]
    )

    chunk_batches.append(batch_text)

print(f"Total batches: {len(chunk_batches)}")


def generate_hypothetical_questions(batch_text):

    response = client.chat.completions.create(
        model=model_name,
        temperature=0,
        max_tokens=300,
        messages=[
            {
                "role": "system",
                "content": """
You will receive multiple document chunks.

Generate ONLY 3 hypothetical questions that best represent
the information across ALL chunks.

Rules:
- Generate exactly 3 questions.
- One question per line.
- No numbering.
- No explanations.
"""
            },
            {
                "role": "user",
                "content": user_message_template.format(
                    documents=batch_text
                )
            }
        ]
    )

    return response.choices[0].message.content


batch_questions = []

for idx, batch_text in enumerate(chunk_batches):

    try:

        questions = generate_hypothetical_questions(batch_text)

        batch_questions.append({
            "batch_id": idx + 1,
            "questions": questions,
            "context": batch_text
        })

        print(f"Processed Batch {idx + 1}")

    except Exception as e:

        print(f"Batch {idx + 1} failed: {e}")

Total batches: 34
Processed Batch 1
Processed Batch 2
Processed Batch 3
Processed Batch 4
Processed Batch 5
Batch 6 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kt0wghx4eb6sdp4t6n08fkwf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 194012, Requested 7114. Please try again in 8m6.431999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Batch 7 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kt0wghx4eb6sdp4t6n08fkwf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 194012, Requested 6758. Please try again in 5m32.64s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Batch 8 failed: Error code: 429 - {'error': 

In [29]:
batch_questions

[]

In [51]:
len(batch_questions)

31

In [52]:
hypothetical_questions_vectorstore = Chroma(
    collection_name="hypothetical_questions",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [53]:
chromadb_client.list_collections()

['hypothetical_questions', 'tesla-10k-2019-to-2023']

In [58]:
from langchain_core.documents import Document

hypothetical_documents = []

for item in batch_questions:

    hypothetical_documents.append(
        Document(
            page_content=item["questions"],
            metadata={
                "batch_id": item["batch_id"]
            }
        )
    )

In [60]:
print(type(hypothetical_documents[0]))

<class 'langchain_core.documents.base.Document'>


In [62]:
hypothetical_questions_vectorstore.add_documents(
    documents=hypothetical_documents)

['4cc299c6-144d-4181-b60a-f4b5ef0ffaa5',
 '92d05aa2-cae1-45fc-92ef-31c5b5683456',
 '20eba623-b416-4948-bd51-13cd1e9262c3',
 '017f53e9-b8e8-48ae-a005-8431e4ad5599',
 '7176a6f1-2094-4bfc-9258-bbcbe56b664e',
 '32328782-4112-4717-bbc0-bf4584a51e24',
 'd5e72c7c-1355-437b-bc09-7a88462a1c45',
 'e000429d-bcf3-4a7f-8b7a-feb229c91d71',
 'df69544f-0192-444b-bad4-c25643dd3ba4',
 '1c138c50-90a9-40e2-9196-079dc905c631',
 '5809d3cc-77fc-4372-b8cc-aeccd7038d27',
 '3f4a6237-e76d-4bac-a0f5-b7ac1540af4f',
 '9ed1936b-56f7-4d25-8c71-33610e1ed1f9',
 '5194266c-4680-4983-9425-c387bcc226b9',
 '91d09ad8-d97b-4734-89ae-e3de33409c20',
 '56ea8c2f-8099-453c-8b06-fb81cfad65b5',
 '3eeada20-a281-4d98-b43f-9e018b0d5280',
 'ae8c69a9-7c65-4487-995b-31d02fe56b6b',
 '5dd59c9a-8949-4f4f-862c-3426dcd6dfea',
 'b865382f-da96-4c39-a709-32213919c978',
 '798a8c01-60f1-40be-bad8-54a6b0865a34',
 '0b19340f-2c77-4dfd-8af3-b9b10cfda081',
 '23763ead-a330-4175-a05e-100a8085512e',
 'c0494837-ec3c-48d1-819b-09ecbcbcd867',
 '5f69afd4-4707-

In [63]:
retriever = hypothetical_questions_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 5}
)

In [64]:
hypothetical_questions_retrieved = retriever.invoke(user_query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [65]:
hypothetical_questions_retrieved

[Document(id='1e6dbeb9-f944-454e-a150-5f9b7f3de2d7', metadata={'batch_number': 5, 'parent_collection': 'full_document_chunks'}, page_content='What are the main components of the cost of automotive sales revenue?'),
 Document(id='9a0fd0a7-97a0-4eb1-a31b-d4757b0c824b', metadata={'batch_number': 11, 'parent_collection': 'full_document_chunks'}, page_content='What are the primary factors contributing to the increase in cost of automotive sales revenue from idle capacity charges in 2020?'),
 Document(id='e048fdf0-b83c-4d8d-9fc2-d5d65a789435', metadata={'batch_number': 25, 'parent_collection': 'full_document_chunks'}, page_content='What is the percentage of digital assets converted into fiat currency as of December 31, 2022?'),
 Document(id='85563296-1b36-476e-b487-822ffc084a7b', metadata={'batch_number': 25, 'parent_collection': 'full_document_chunks'}, page_content='What is the total amount of digital assets purchased and/or received by the company during the years ended December 31, 2022 

In [67]:
print(hypothetical_questions_retrieved[0].metadata)

{'batch_number': 5, 'parent_collection': 'full_document_chunks'}


In [68]:
BATCH_SIZE = 100

retrieved_documents = []

for doc in hypothetical_questions_retrieved:

    batch_num = doc.metadata["batch_number"]

    start_idx = (batch_num - 1) * BATCH_SIZE
    end_idx = min(
        start_idx + BATCH_SIZE,
        len(tesla_10k_chunks)
    )

    retrieved_documents.extend(
        tesla_10k_chunks[start_idx:end_idx]
    )

print(f"Retrieved {len(retrieved_documents)} chunks")

Retrieved 500 chunks


In [69]:
print(retrieved_documents)

[Document(metadata={'producer': 'Qt 5.11.3', 'creator': 'wkhtmltopdf 0.12.5', 'creationdate': '2020-09-30T05:19:35+00:00', 'title': '', 'source': 'tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf', 'total_pages': 297, 'page': 80, 'page_label': '81'}, page_content='accounted\tfor\tas\tleases,\tno\tlonger\tmeet\tthe\tdefinition\tof\ta\tlease\tand\tare\tinstead\taccounted\tfor\tin\taccordance\twith\tthe\tnew\trevenue\tstandard\n(refer\tto\tthe\t\nLeases\n\tsection\tbelow\tfor\tdetails).\n\t\nCost\tof\tRevenues\nAutomotive\tSegment\nAutomotive\tSales\nCost\tof\tautomotive\tsales\trevenue\tincludes\tdirect\tparts,\tmaterial\tand\tlabor\tcosts,\tmanufacturing\toverhead,\tincluding\tdepreciation\tcosts\tof\ntooling\tand\tmachinery,\tshipping\tand\tlogistic\tcosts,\tvehicle\tconnectivity\tcosts,\tallocations\tof\telectricity\tand\tinfrastructure\tcosts\trelated\tto\tour\nSupercharger\tnetwork,\tand\treserves\tfor\testimated\twarranty\texpenses.\tCost\tof\tautomotive\tsales\trevenues\talso\tin

In [71]:
print(type(retrieved_documents))

<class 'list'>


In [72]:
print(type(retrieved_documents[0]))

<class 'langchain_core.documents.base.Document'>


In [73]:
retrieved_context = "\n\n".join(
    doc.page_content
    for doc in retrieved_documents
)

In [74]:
retrieved_context

'accounted\tfor\tas\tleases,\tno\tlonger\tmeet\tthe\tdefinition\tof\ta\tlease\tand\tare\tinstead\taccounted\tfor\tin\taccordance\twith\tthe\tnew\trevenue\tstandard\n(refer\tto\tthe\t\nLeases\n\tsection\tbelow\tfor\tdetails).\n\t\nCost\tof\tRevenues\nAutomotive\tSegment\nAutomotive\tSales\nCost\tof\tautomotive\tsales\trevenue\tincludes\tdirect\tparts,\tmaterial\tand\tlabor\tcosts,\tmanufacturing\toverhead,\tincluding\tdepreciation\tcosts\tof\ntooling\tand\tmachinery,\tshipping\tand\tlogistic\tcosts,\tvehicle\tconnectivity\tcosts,\tallocations\tof\telectricity\tand\tinfrastructure\tcosts\trelated\tto\tour\nSupercharger\tnetwork,\tand\treserves\tfor\testimated\twarranty\texpenses.\tCost\tof\tautomotive\tsales\trevenues\talso\tincludes\tadjustments\tto\twarranty\texpense\nand\tcharges\tto\twrite\tdown\tthe\tcarrying\tvalue\tof\tour\tinventory\twhen\tit\texceeds\tits\testimated\tnet\trealizable\tvalue\tand\tto\tprovide\tfor\tobsolete\tand\ton-\nhand\tinventory\tin\texcess\tof\tforecasted\td

In [75]:
len(retrieved_context)

588237

In [77]:
qna_system_message1 = """
You are a helpful AI assistant.

Answer the user's question using only the information provided in the context.

Rules:
- Use only the provided context.
- Do not make up information.
- If the answer is not available in the context, say:
  "I could not find the answer in the provided context."
- Be concise and accurate.
"""
qna_user_message_template1 = """
Context:
{context}

Question:
{question}

Answer:
"""

final_prompt = [
    {
        "role": "system",
        "content": qna_system_message1
    },
    {
        "role": "user",
        "content": qna_user_message_template1.format(
            context=retrieved_context,
            question=user_query
        )
    }
]

response = client.chat.completions.create(
    model=model_name,
    messages=final_prompt,
    temperature=0
)

final_answer = response.choices[0].message.content
print(final_answer)

$ 44,125
